# `align_bundle` — yoke behaviour to the t-series time base

**The t-series defines time.** Its frame grid is the output grid; dF/F is never
resampled. The behaviour camera runs slower (~15 fps vs 30–50 Hz), so box mean
intensity and motion energy are **interpolated up** onto the 2P frames.

**Alignment and interpolation only.** No correlations, no bout detection, no sniff.

## What you supply, per run — five values, all hardcoded
| | |
|---|---|
| `ca` | folder holding `dff.npy` — the t-series output |
| `tseries_fps` | **the 2P frame rate. This sets the time axis.** |
| `beh` | the behaviour `_boxtraces.npz`, or the `proc/` folder holding it |
| `on` / `off` | first and last camera frame with the laser on |

Fully filled in for the two Thor runs — nothing to edit, just run it. `brukermouse1` is
commented out in the config: v6 extracted 8179 of that t-series' 18259 frames, so its
`dff` covers only the first 4.5 min of a 10 min recording. Its anchors are correct and it
can be re-enabled by uncommenting once v6 has been re-run on the full movie.

## The mapping — two modes, chosen by `cam_fps`

```
t_2p = k / tseries_fps                    the t-series frame grid
D    = t_2p[-1] + P = n_2p / fps          full t-series span, s
```

**`cam_fps = None` → two-anchor.** `on` and `off` pin both ends, so the camera rate falls
out of the map and drift between the ThorCam and microscope oscillators is absorbed:

```
t_cam(f) = (f - on) * D / (off - on)
```

Requires `off` to be the true laser-OFF frame — i.e. the camera outlasted the t-series.

**`cam_fps = <Hz>` → single-anchor.** `on` alone sets the origin and the supplied rate
sets the scale; `off` only marks where coverage stops:

```
t_cam(f) = (f - on) / cam_fps
```

Use this whenever the second anchor is not the end of the t-series — the camera stopped
early, or `dff` is a prefix of the recording. **2P frames past the end of camera coverage
are dropped, never extrapolated**, so `t`, `dff` and `beh_*` stay the same length and
NaN-free; `n_2p_dropped` and `coverage_end_s` record what went.

Both modes print the rate the two anchors *would* have implied, so the disagreement is
always visible.

## ⚠ `tseries_fps` sets the scale of everything
Get it wrong and the whole time axis stretches uniformly — every trace still looks
plausible. Two things would show it, and only the first stops the run:

1. **`params.json`** — when the `ca` folder has one, v6 recorded the fps it actually
   used. A mismatch with your typed `tseries_fps` **refuses the run**, because that is
   a flat contradiction between two records of the same recording.
2. **Implied camera rate** — `(off − on) / D`, printed per run. Derived, never assumed,
   and **never enforced**: camera rates vary between recordings, so there is no sane
   fixed bound to check against. A 2× error in `tseries_fps` halves or doubles this
   number, so it is worth a glance — but it is your call, not the notebook's.

## Accuracy
- **origin ±1 camera frame**, common-mode — the frame straddling the laser edge is
  bright for an unknown fraction of its exposure. Recorded per run as
  `origin_uncertainty_s`, since the camera period varies between recordings.
- **scale ±1/(off−on)** ≈ ±115 ppm at 8 800 bright frames ≈ ±0.07 s over 10 min.

Good to ~±0.1 s absolute. Enough for envelope-vs-dF/F work; **not** enough to claim
frame-level ordering of behavioural onsets against neural events.

In [ ]:
# ========================= THE ONLY CELL YOU EDIT =========================
from pathlib import Path

OUT_DIR = Path("/grid/courses/data/imagcourse/GECI_Project_Analyzed/bundles")

# --- THE T-SERIES DEFINES THE TIME BASE ---------------------------------
# ca          : folder holding dff.npy (the t-series output)
# tseries_fps : 2P frame rate, Hz. THIS SETS THE TIME AXIS -- see the guards above.
#
# --- THE BEHAVIOUR CAMERA IS YOKED TO IT --------------------------------
# beh     : the behaviour _boxtraces.npz, or the proc/ folder holding it
# on      : FIRST camera frame with the laser ON  (t = 0)
# off     : LAST usable camera frame -- the laser-OFF frame if the camera outlasted the
#           t-series, otherwise simply the last frame the camera recorded.
# cam_fps : camera rate, Hz. Sets which of the two mappings is used:
#             None  -> TWO-ANCHOR. on and off pin both ends; the rate is derived and any
#                      camera/microscope drift is absorbed. Requires off to be the true
#                      laser-OFF frame.
#             value -> SINGLE-ANCHOR. on plus this rate define the map; off only marks
#                      where coverage stops. Use this whenever the camera stopped before
#                      the t-series ended, or the dff is a prefix of the recording.
#           2P frames past the end of camera coverage are DROPPED, never extrapolated.

DATA = "/grid/courses/data/imagcourse/GECI Project Jonathons/Data to Analyze"
ANALYZED = "/grid/courses/data/imagcourse/GECI_Project_Analyzed"

RUNS = {
    "thormouse1_spont": dict(
        ca          = f"{ANALYZED}/thor1_mouse1_spontaneous1",
        tseries_fps = 30.0,
        beh         = f"{DATA}/20260807_M1_Mouse1_Thor/Mouse 1_spontaneous trial"
                      f"/Mouse1_spontaneous_trial1_behavior/proc",
        on          = 119,
        off         = 28778,    # camera STOPPED ~4 min before the t-series ended
        cam_fps     = 30.0,     # -> single-anchor; last 7341 of 36000 2P frames uncovered
    ),
    "thormouse2_spont": dict(
        ca          = f"{ANALYZED}/thor2_mouse2_spontaneous2",
        tseries_fps = 30.0,
        beh         = f"{DATA}/20260807_M1_Mouse2_Thor/Mouse2_spontaneous_trial"
                      f"/Mouse2_spon_behavior1/proc",
        on          = 61,
        off         = 13960,
        cam_fps     = 15.0,
    ),
    # DROPPED 2026-08-10 (Seneca): v6 extracted 8179 of the t-series' 18259 frames, so
    # dff covers only the first 4.5 min of a 10 min recording. The anchors and rates below
    # are correct -- against the full 18259 frames they imply 14.77 Hz vs a stated 15, with
    # the camera stopping 9.4 s early. Re-run v6 on that movie (check whether it is split
    # across several tifs: find_movie takes the largest single one) and uncomment.
    # "brukermouse1_spont": dict(
    #     ca          = f"{ANALYZED}/TSeries-08082026-0955-020",
    #     tseries_fps = 30.0,
    #     beh         = f"{DATA}/20260808_M1_Mouse1/spon_sniff_run1/behavior/proc",
    #     on          = 148,
    #     off         = 9136,
    #     cam_fps     = 15.0,
    # ),
}

ENV_S            = 0.33   # conditioned-envelope window, s. Matches whisk_bouts.
SAVE_MAT         = True   # also write .mat (non-fatal if scipy is missing)
REPO_DIR         = None   # folder containing whisk_bouts.py. None = autodetect.

In [ ]:
import json, sys, traceback
from pathlib import Path            # self-sufficient: do not rely on the config cell
import numpy as np

for _need in ("RUNS", "OUT_DIR", "ENV_S", "SAVE_MAT", "REPO_DIR"):
    if _need not in globals():
        raise NameError(f"{_need} is not defined -- run the config cell above first "
                        f"(kernel restarted?).")

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("output ->", OUT_DIR)

# condition() is imported, never reimplemented: two copies of a filter drift apart.
# If it cannot be found, beh_env is simply absent -- raw ME and luminance are unaffected.
_cand = [REPO_DIR] if REPO_DIR else []
_cand += [Path.cwd(), Path.cwd().parent, Path.cwd() / "behavioral motif analysis",
          Path.cwd().parent / "behavioral motif analysis"]
condition = None
for _d in _cand:
    if _d and (Path(_d) / "whisk_bouts.py").exists():
        sys.path.insert(0, str(Path(_d)))
        from whisk_bouts import condition            # noqa: E402
        print(f"condition() imported from {Path(_d)/'whisk_bouts.py'}")
        break
if condition is None:
    print("NOTE: whisk_bouts.py not found -> 'beh_env' will be ABSENT from the bundles.\n"
          "      Raw motion energy and luminance are unaffected. Set REPO_DIR to fix.")

In [ ]:
def find_boxtraces(path):
    """Accept either the npz itself or the proc/ folder holding it. The glob excludes
    '*_boxtraces_with_time.npz', which is a different file with a different contract."""
    p = Path(path)
    if p.is_file():
        return p
    if not p.is_dir():
        raise FileNotFoundError(f"behaviour path not found: {p}")
    hits = sorted(q for q in p.glob("*_boxtraces.npz") if q.is_file())
    if not hits:
        have = sorted(q.name for q in p.glob("*.npz"))
        raise FileNotFoundError(f"{p}: no *_boxtraces.npz. npz files present: {have}")
    if len(hits) > 1:
        raise ValueError(f"{p}: {len(hits)} candidates {[q.name for q in hits]} -- name "
                         f"the file explicitly in the config rather than guessing.")
    return hits[0]


def load_behaviour(npz_path):
    """_boxtraces.npz (or its proc/ folder) -> dict. Handles both vintages (older files
    lack trim/truncation keys) and never indexes boxes by position -- box sets differ."""
    p = find_boxtraces(npz_path)
    if p.stat().st_size < 1024:
        raise ValueError(f"{p.name}: {p.stat().st_size} bytes -- this file did not "
                         f"upload. Re-fetch it before aligning.")
    d = np.load(p, allow_pickle=True)
    keys = set(d.files)
    for req in ("motion_energy", "traces", "box_names"):
        if req not in keys:
            raise KeyError(f"{p.name}: missing '{req}' (keys: {sorted(keys)})")

    names = [str(n) for n in d["box_names"]]
    me = np.asarray(d["motion_energy"], dtype=np.float64)     # n_box x n_cam
    lum = np.asarray(d["traces"], dtype=np.float64)
    if me.shape != lum.shape:
        raise ValueError(f"{p.name}: motion_energy {me.shape} != traces {lum.shape}")
    if me.shape[0] != len(names):
        raise ValueError(f"{p.name}: {me.shape[0]} rows but {len(names)} box names")

    return dict(names=names, me=me, lum=lum, n_cam=me.shape[1], path=str(p),
                truncated=bool(d["truncated"]) if "truncated" in keys else False,
                trim_tail=int(d["trim_tail"]) if "trim_tail" in keys else 0,
                vintage="aug9+" if "trim_tail" in keys else "pre-trim")


def load_neural(folder):
    """v6 notebook output (loose .npy) or ca_extract's ca.npz. dF/F is [n_roi x n_2p]:
    v6 computes F0 with axis=1 and the FOV mean with axis=0, so rows are ROIs."""
    f = Path(folder)
    if not f.exists():
        raise FileNotFoundError(f"neural folder not found: {f}")

    out = {"path": str(f)}
    if (f / "dff.npy").exists():                                  # ---- v6 layout
        out["layout"] = "v6_npy"
        if not (f / "_COMPLETE").exists():
            print(f"  *** WARNING: no _COMPLETE in {f.name} -- that v6 run may not have "
                  f"finished, and its arrays may be left over from a previous run. ***")
        out["dff"] = np.load(f / "dff.npy")
        for key, fn in (("F_raw", "traces_raw.npy"), ("roi_npix", "roi_npix.npy"),
                        ("F0", "F0.npy")):
            if (f / fn).exists():
                out[key] = np.load(f / fn)
        if (f / "params.json").exists():
            out["params"] = json.loads((f / "params.json").read_text())
        if (f / "roi_map.tif").exists():                          # centroids: v6 saves none
            try:
                import tifffile as tiff
                rm = tiff.imread(f / "roi_map.tif")
                lab = [l for l in np.unique(rm) if l != 0]
                out["roi_xy"] = np.array([np.argwhere(rm == l).mean(0)[::-1] for l in lab],
                                         dtype=np.float64) if lab else np.zeros((0, 2))
            except Exception as e:
                print(f"  (roi_map.tif unreadable, roi_xy skipped: {e})")
    elif (f / "ca.npz").exists():                                 # ---- ca_extract layout
        out["layout"] = "ca_npz"
        z = np.load(f / "ca.npz", allow_pickle=True)
        out["dff"] = z["dff"]
        for k in ("F_raw", "roi_npix", "roi_xy", "F0", "frame_times", "fps"):
            if k in z.files:
                out[k] = z[k]
    else:
        raise FileNotFoundError(f"{f}: no dff.npy and no ca.npz")

    dff = np.asarray(out["dff"], dtype=np.float64)
    if dff.ndim != 2:
        raise ValueError(f"{f}: dff has shape {dff.shape}, expected 2-D")
    if dff.shape[0] > dff.shape[1]:
        raise ValueError(
            f"{f}: dff is {dff.shape[0]} x {dff.shape[1]} -- more ROIs than frames. "
            f"v6 writes [n_roi x n_frames]; this looks TRANSPOSED. Refusing to guess.")
    out["dff"], out["n_roi"], out["n_2p"] = dff, dff.shape[0], dff.shape[1]
    return out

In [ ]:
def build_tseries_clock(cfg, C, name):
    """The t-series time base: frame times + the full span D, in seconds.

    Returns (t_2p [n_2p], P, D, info). The rate is the hardcoded one. If the neural
    file happens to carry measured per-frame times (ca_extract's ca.npz does; v6's
    loose .npy do not) they are used instead -- they absorb frame-period jitter, and
    they cost nothing to read since they are already in the file being loaded.
    """
    fps, n_2p = float(cfg["tseries_fps"]), C["n_2p"]
    info = {"fps_typed": fps, "clock": "constant_rate", "notes": []}

    ft = C.get("frame_times")
    if ft is not None and len(np.asarray(ft)) == n_2p:
        t_2p = np.asarray(ft, dtype=np.float64)
        t_2p = t_2p - t_2p[0]                         # relativeTime does not start at 0
        P = float(np.median(np.diff(t_2p)))
        info["clock"] = "measured_frame_times"
        if abs(1 / P - fps) / fps > 0.01:
            raise ValueError(f"{name}: tseries_fps={fps} but the file's own frame_times "
                             f"imply {1/P:.4f} Hz. Resolve before aligning.")
    else:
        P = 1.0 / fps
        t_2p = np.arange(n_2p) * P

    prm = C.get("params") or {}
    if prm.get("fps") and abs(prm["fps"] - fps) / fps > 0.01:
        raise ValueError(f"{name}: tseries_fps={fps} but v6 params.json says "
                         f"fps={prm['fps']:.4f}. Resolve before continuing.")
    if prm.get("n_frames") and int(prm["n_frames"]) != n_2p:
        print(f"    *** WARNING: params.json n_frames={prm['n_frames']} but dff has "
              f"{n_2p} columns. ***")

    D = float(t_2p[-1] + P)          # full span: last frame ONSET plus its own period
    return t_2p, P, D, info


def align(name, cfg):
    """Interpolate behaviour up onto the t-series frame grid. Returns arrays + meta;
    raises on anything that would produce a quietly-wrong bundle."""
    for k in ("ca", "beh", "on", "off", "tseries_fps"):
        if cfg.get(k) is None:
            raise ValueError(f"{name}: '{k}' is None -- fill it in the config cell")
    on, off = int(cfg["on"]), int(cfg["off"])

    B = load_behaviour(cfg["beh"])
    C = load_neural(cfg["ca"])
    n_cam, n_roi, n_2p = B["n_cam"], C["n_roi"], C["n_2p"]

    # ---- the t-series clock: this defines time -------------------------------
    t_2p, P, D, clk = build_tseries_clock(cfg, C, name)

    # ---- anchors -------------------------------------------------------------
    if not (0 <= on < off <= n_cam - 1):
        raise ValueError(f"{name}: need 0 <= on < off <= {n_cam-1}, got on={on} off={off}")
    span = off - on                                   # intervals, not frames

    # ---- yoke the camera to it ----------------------------------------------
    cam_fps_implied = span / D                        # what two anchors alone would give
    cam_fps_cfg = cfg.get("cam_fps")
    if cam_fps_cfg:                                   # SINGLE-ANCHOR: on + a known rate
        mode = "single_anchor_known_fps"
        cam_dt = 1.0 / float(cam_fps_cfg)
    else:                                             # TWO-ANCHOR: on and off pin both ends
        mode = "two_anchor"
        cam_dt = D / span
    cam_fps_used = 1.0 / cam_dt
    t_cam = (np.arange(n_cam) - on) * cam_dt          # defined for ALL camera frames

    print(f"    t-series {clk['clock']}  {1/P:.4f} Hz x {n_2p} frames = {D:.2f} s")
    print(f"    camera   {n_cam} frames, {on}..{off} ({span} intervals), mode={mode}")
    print(f"    cam rate used {cam_fps_used:.4f} Hz"
          + (f"   (two anchors alone would imply {cam_fps_implied:.4f} Hz"
             f", {100*(cam_fps_implied/cam_fps_used - 1):+.1f}%)"
             if mode == "single_anchor_known_fps" else ""))

    # ---- coverage: 2P frames past the camera are DROPPED, never extrapolated -
    off_eff = min(off, n_cam - 1)
    t_cov_end = float(t_cam[off_eff])
    keep = int(np.searchsorted(t_2p, t_cov_end, side="right"))
    n_2p_full, dropped_2p = n_2p, n_2p - keep
    if keep < 2:
        raise ValueError(f"{name}: camera coverage ends at {t_cov_end:.2f} s, before the "
                         f"second 2P frame. Check `on`, `off` and cam_fps.")
    if dropped_2p:
        print(f"    *** camera coverage ends at {t_cov_end:.2f} s = 2P frame {keep-1}. "
              f"DROPPING the last {dropped_2p} of {n_2p} 2P frames "
              f"({(D - t_cov_end)/60:.1f} min) -- no behaviour exists for them. ***")
        t_2p, n_2p = t_2p[:keep], keep
        C = {**C, "dff": C["dff"][:, :keep]}
        if "F_raw" in C:
            C["F_raw"] = np.asarray(C["F_raw"])[:, :keep]

    up = P / cam_dt                                   # interpolation factor
    print(f"    behaviour interpolated {'UP' if up < 1 else 'DOWN'} x{1/up:.2f} "
          f"onto {n_2p} t-series frames")

    if up > 1.0:
        print(f"    *** WARNING: the t-series ({1/P:.2f} Hz) is SLOWER than the camera "
              f"({cam_fps_used:.2f} Hz). Linear interpolation then decimates without "
              f"anti-aliasing -- behavioural power above {0.5/P:.2f} Hz will fold. ***")

    # ---- repair the two structurally-invalid ME samples ----------------------
    me = B["me"].copy()
    if n_cam > 1:
        me[:, 0] = me[:, 1]          # frame 0 is 0 by construction (no previous frame)
    if on + 1 <= off:
        me[:, on] = me[:, on + 1]    # frame `on` holds the dark->bright laser step
    lum = B["lum"]

    # ---- no extrapolation, ever ---------------------------------------------
    if not (t_cam[0] <= t_2p[0] and t_2p[-1] <= t_cam[-1]):
        raise ValueError(f"{name}: t-series grid [{t_2p[0]:.3f}, {t_2p[-1]:.3f}] is not "
                         f"inside the camera span [{t_cam[0]:.3f}, {t_cam[-1]:.3f}]")

    interp = lambda A: np.vstack([np.interp(t_2p, t_cam, row) for row in A])
    beh_me, beh_lum = interp(me), interp(lum)

    beh_env = None
    if condition is not None:
        # condition the FULL camera trace, then interpolate: moving_average pads by edge
        # replication, so conditioning a pre-cropped trace fabricates its endpoints.
        beh_env = interp(np.vstack([condition(row, cam_fps_used, ENV_S) for row in me]))

    artifact = (t_2p < cam_dt) | (t_2p > t_2p[-1] - cam_dt)   # the two coverage edges

    blocks = [("roi", C["dff"], [f"roi_{i:04d}" for i in range(n_roi)]),
              ("me", beh_me, [f"me_{b}" for b in B["names"]]),
              ("lum", beh_lum, [f"lum_{b}" for b in B["names"]])]
    if beh_env is not None:
        blocks.append(("env", beh_env, [f"env_{b}" for b in B["names"]]))
    M = np.vstack([b[1] for b in blocks])
    M_names = [n for b in blocks for n in b[2]]

    meta = dict(run=name, on=on, off=off, span_intervals=span,
                tseries_fps=cfg["tseries_fps"], tseries_frame_period_s=P,
                tseries_duration_s=D, tseries_clock=clk["clock"],
                tseries_notes=clk["notes"],
                map_mode=mode, cam_fps_used=cam_fps_used,
                cam_fps_implied_by_anchors=cam_fps_implied,
                cam_fps_config=cam_fps_cfg, interp_factor=1.0 / up,
                coverage_end_s=t_cov_end, n_2p_full=n_2p_full, n_2p_dropped=dropped_2p,
                n_cam=n_cam, n_2p=n_2p, n_roi=n_roi, n_box=len(B["names"]),
                box_names=B["names"], origin_uncertainty_s=cam_dt,
                scale_uncertainty_ppm=1e6 / span,
                beh_npz=B["path"], beh_vintage=B["vintage"], beh_truncated=B["truncated"],
                beh_trim_tail=B["trim_tail"], ca_folder=C["path"], ca_layout=C["layout"],
                ca_params=C.get("params") or {},
                env_s=ENV_S if beh_env is not None else None,
                conditioned=beh_env is not None,
                notes=[f"map mode: {mode}",
                       "the t-series frame grid IS the output grid",
                       "t=0 is the first camera frame with the laser on",
                       "dff is NOT resampled and NOT z-scored",
                       "me/lum/env are interpolated onto the t-series grid (linear)",
                       "me[:,0] and me[:,on] repaired before interpolation"])

    arrays = dict(t=t_2p, dff=C["dff"], beh_me=beh_me, beh_lum=beh_lum,
                  beh_names=np.array(B["names"]), artifact=artifact,
                  M=M, M_names=np.array(M_names))
    if beh_env is not None:
        arrays["beh_env"] = beh_env
    for k in ("F_raw", "roi_npix", "roi_xy"):
        if k in C:
            arrays[k] = np.asarray(C[k])

    bad = {k: v for k, v in arrays.items()
           if v.dtype.kind == "f" and not np.isfinite(v).all()}
    if bad:
        raise ValueError(f"{name}: non-finite values in {sorted(bad)}")
    return arrays, meta

In [ ]:
results, failed = {}, {}
for name, cfg in RUNS.items():
    print(f"\n=== {name} ===")
    try:
        arrays, meta = align(name, cfg)

        stem = OUT_DIR / f"{name}_aligned"
        tmp = stem.with_suffix(".npz.tmp")                  # atomic: never a half file
        with open(tmp, "wb") as fh:
            np.savez_compressed(fh, meta=json.dumps(meta), **arrays)
        tmp.replace(stem.with_suffix(".npz"))
        stem.with_suffix(".json").write_text(json.dumps(meta, indent=2))
        if SAVE_MAT:
            try:
                from scipy.io import savemat
                savemat(stem.with_suffix(".mat"),
                        {**{k: v for k, v in arrays.items()}, "meta": json.dumps(meta)})
            except Exception as e:
                print(f"    (.mat skipped: {e})")

        results[name] = (arrays, meta)
        print(f"    wrote {stem.with_suffix('.npz').name}   M {arrays['M'].shape}  "
              f"({meta['n_roi']} ROIs + {len(arrays['beh_names'])} boxes x "
              f"{'3' if 'beh_env' in arrays else '2'} channels) x {meta['n_2p']} frames")
    except Exception as e:
        failed[name] = f"{type(e).__name__}: {e}"
        print(f"    FAILED -- {failed[name]}")
        traceback.print_exc()                    # one bad run must not cost the others

print("\n" + "=" * 86)
print(f"{'run':20s} {'2P Hz':>7s} {'mode':>24s} {'cam Hz':>8s} {'T':>7s} "
      f"{'dropped':>8s} {'n_roi':>6s} {'cover s':>8s}")
for n, (_, m) in results.items():
    print(f"{n:20s} {1/m['tseries_frame_period_s']:7.3f} {m['map_mode']:>24s} "
          f"{m['cam_fps_used']:8.4f} {m['n_2p']:7d} {m['n_2p_dropped']:8d} "
          f"{m['n_roi']:6d} {m['coverage_end_s']:8.1f}")
for n, e in failed.items():
    print(f"{n:20s} FAILED: {e}")
print("=" * 86)
print("'dropped' = 2P frames past the end of camera coverage. They are removed from the\n"
      "bundle, never extrapolated, so t/dff/beh_* all stay the same length and NaN-free.\n"
      "In two_anchor mode 'cam Hz' is derived from the anchors; in single_anchor mode it\n"
      "is the rate you supplied, and the anchor-implied rate is printed above for contrast.")

In [ ]:
# QC -- the eyeball gate. Top: the anchors on the laser trace. Bottom: 60 s of the
# aligned product. Look at both before anything downstream touches these bundles.
import matplotlib.pyplot as plt

for name, (A, m) in results.items():
    B = load_behaviour(RUNS[name]["beh"])
    lb = "laser_trigger" if "laser_trigger" in B["names"] else B["names"][0]
    lt = B["lum"][B["names"].index(lb)]

    fig, ax = plt.subplots(2, 1, figsize=(13, 6))
    ax[0].plot(lt, lw=.5, color="0.3")
    ax[0].axvline(m["on"], color="g", lw=1.2, label=f"on {m['on']}")
    ax[0].axvline(m["off"], color="r", lw=1.2, label=f"off {m['off']}")
    ax[0].set(title=f"{name} -- '{lb}' luminance, camera clock   "
                    f"{m['map_mode']}, cam {m['cam_fps_used']:.3f} Hz",
              xlabel="camera frame", ylabel="mean intensity")
    ax[0].legend(loc="upper right", fontsize=8)

    t, sl = A["t"], slice(0, min(len(A["t"]), int(60 / m["tseries_frame_period_s"])))
    src = A.get("beh_env", A["beh_me"])
    tag = "env" if "beh_env" in A else "raw ME"
    for i, b in enumerate(A["beh_names"]):
        y = src[i][sl]
        rng = np.ptp(y)
        ax[1].plot(t[sl], (y - y.mean()) / (rng if rng else 1) + i, lw=.7, label=str(b))
    d = A["dff"][:, sl]
    ax[1].plot(t[sl], d.mean(0) / (np.ptp(d.mean(0)) or 1) - 1.2, lw=.9, color="k",
               label="mean dF/F")
    ax[1].set(title=f"first 60 s on the t-series grid ({tag}, offset for display; "
                    f"behaviour interpolated {m['interp_factor']:.2f}x)",
              xlabel="s since t-series start", yticks=[])
    ax[1].legend(fontsize=7, ncol=4, loc="upper right")
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"{name}_qc.png", dpi=140, bbox_inches="tight")
    plt.show()

## Output contract

`<OUT_DIR>/<run>_aligned.npz` — everything below shares one time axis, `t`, and that
axis is the t-series frame grid.

| key | shape | |
|---|---|---|
| `t` | `[T]` | seconds since t-series start; `T = n_2p`; measured frame times where the rig recorded them |
| `dff` | `[n_roi × T]` | **untouched** — not resampled, not z-scored |
| `beh_me` | `[n_box × T]` | raw motion energy, interpolated up |
| `beh_lum` | `[n_box × T]` | box mean intensity, interpolated up |
| `beh_env` | `[n_box × T]` | conditioned log-ME envelope — **absent if `whisk_bouts` was not importable** |
| `beh_names` | `[n_box]` | verbatim from `box_extract`; **index by name, never position** |
| `artifact` | `[T]` bool | frames straddling a laser transition |
| `M` | `[(n_roi + k·n_box) × T]` | the aligned matrix: `vstack(dff, beh_me, beh_lum[, beh_env])` |
| `M_names` | `[n_roi + k·n_box]` | `roi_0000` / `me_<box>` / `lum_<box>` / `env_<box>` |
| `F_raw`, `roi_npix`, `roi_xy` | | carried when the neural folder has them |
| `meta` | JSON string | anchors, t-series clock + source, implied camera rate, interp factor, uncertainties, provenance |

```python
z = np.load("thormouse1_spont_aligned.npz", allow_pickle=True)
t, dff, M = z["t"], z["dff"], z["M"]
names = [str(s) for s in z["beh_names"]]
whisk = z["beh_env"][names.index("whisker_pad")]     # by name
meta  = json.loads(str(z["meta"]))
```

## Two things to carry downstream

**Upsampling created no information.** The behavioural degrees of freedom are still at the
camera rate — `meta["interp_factor"]` says by how much they were stretched. Any per-frame
statistic computed on `T` samples will look about that many times more significant than
it is.

**`beh_me` is raw motion energy** and carries flicker out to the camera Nyquist (56% of
suprathreshold runs last a single frame). `beh_env` — log + zero-phase 330 ms envelope —
is the band-limited version and the right regressor for anything correlational. Both are
in the file; the choice is yours, but it is a choice.